# Solubility

In [1]:
import json, numpy as np, pandas as pd
from scipy import stats
d = pd.read_csv('chi_parameter.csv')
for c in ['Polymer', 'Solvent', 'Polymer_SMILES', 'Solvent_SMILES']: d[c] = d[c].astype(str).str.strip()
d['Polymer'] = d.Polymer.str.replace('µ', 'ε')
d['src'] = d.Reference.map(lambda r: 'Orwoll & Arnold 2007 (Handbook ch.14)' if '69002-5_14' in r else 'Polymer Handbook 4th ed.' if 'Brandrup' in r else 'Polymer 1980 (0032-3861(80)90035-X)' if '90035' in r else 'other')
rows = []
for (p, s), g in d.groupby(['Polymer', 'Solvent']):
    if g.temperature.nunique() < 3: continue
    x = 1 / g.temperature.values; y = g.chi.values; n = len(y)
    r = stats.linregress(x, y)
    Thm = n / np.sum(1 / g.temperature.values)
    resid = y - (r.intercept + r.slope * x); s2 = np.sum(resid ** 2) / (n - 2) if n > 2 else np.nan
    Sxx = np.sum((x - x.mean()) ** 2)
    rows.append(dict(polymer=p, solvent=s, n=n, nref=g.Reference.nunique(), srcs='; '.join(sorted(g.src.unique())), Tmin=g.temperature.min(), Tmax=g.temperature.max(), dT=g.temperature.max() - g.temperature.min(),
                     Thm=Thm, A=r.intercept, B=r.slope, R2=r.rvalue ** 2, se_B=np.sqrt(s2 / Sxx) if n > 2 else np.nan,
                     se_A=np.sqrt(s2 * (1 / n + x.mean() ** 2 / Sxx)) if n > 2 else np.nan, chi_hm=r.intercept + r.slope / Thm, chi_mean=y.mean()))
F = pd.DataFrame(rows); F.to_csv('out/fits.csv', index=False)
G = F[F.R2 > 0.7]
O = {}
O['n_pairs_multiT'] = len(F); O['n_R2_07'] = len(G); O['n_R2_09'] = int((F.R2 > 0.9).sum()); O['R2_median'] = F.R2.median()
f = stats.linregress(G.B, G.A); O['repro'] = dict(slope=f.slope, intercept=f.intercept, r=f.rvalue, p=f.pvalue, Tcomp=-1 / f.slope)
O['frac_Bpos'] = float((G.B > 0).mean()); O['n_polymers_G'] = G.polymer.nunique()
O['Thm_harmonic_of_pairs'] = float(len(G) / np.sum(1 / G.Thm)); O['Thm_mean'] = float(G.Thm.mean())
# identity: A = chi_hm - B/Thm
O['identity_maxdev'] = float(np.max(np.abs(G.A - (G.chi_hm - G.B / G.Thm))))
O['sd_B_over_Thm'] = float((G.B / G.Thm).std()); O['sd_chi_hm'] = float(G.chi_hm.std())
O['r_chihm_B'] = list(stats.pearsonr(G.chi_hm, G.B)); O['rho_chihm_B'] = list(stats.spearmanr(G.chi_hm, G.B))
# null: permute chi_hm across pairs
rng = np.random.default_rng(0); rn = []; tn = []
for _ in range(10000):
    ch = rng.permutation(G.chi_hm.values); Ast = ch - G.B.values / G.Thm.values
    q = stats.linregress(G.B.values, Ast); rn.append(q.rvalue); tn.append(-1 / q.slope)
O['null_r_mean'] = float(np.mean(rn)); O['null_r_CI'] = [float(np.percentile(rn, 2.5)), float(np.percentile(rn, 97.5))]
O['null_Tcomp_median'] = float(np.median(tn)); O['null_Tcomp_CI'] = [float(np.percentile(tn, 2.5)), float(np.percentile(tn, 97.5))]
O['frac_null_r_le_obs'] = float(np.mean(np.array(rn) <= f.rvalue))
# subsets
def sub(X, lab):
    q = stats.linregress(X.B, X.A); return dict(label=lab, n=len(X), r=q.rvalue, Tcomp=-1 / q.slope, Thm=float(len(X) / np.sum(1 / X.Thm)), r_chihm_B=stats.pearsonr(X.chi_hm, X.B)[0])
O['subsets'] = [sub(G, 'R² > 0.7 (as in draft)'), sub(G[G.dT > 50], 'R² > 0.7, ΔT > 50 K'), sub(G[G.nref == 1], 'R² > 0.7, single source'), sub(F, 'all 143 pairs'),
                sub(F[F.nref == 1], 'all pairs, single source'), sub(F[(F.nref == 1) & (F.dT > 50)], 'single source, ΔT > 50 K')]
# partial corr controlling Tmean (draft's test)
def pcorr(a, b, c):
    ra = a - np.polyval(np.polyfit(c, a, 1), c); rb = b - np.polyval(np.polyfit(c, b, 1), c); return stats.pearsonr(ra, rb)[0]
O['partial_draft'] = pcorr(G.A.values, G.B.values, G.Thm.values)
# variance of chi(T) across pairs vs T
Ts = np.arange(250, 701, 5); v = [np.var(G.A + G.B / T) for T in Ts]
O['var_min_T'] = float(Ts[int(np.argmin(v))]); O['data_T_range'] = [float(G.Tmin.min()), float(G.Tmax.max())]
# per-pair estimation-error correlation of A and B: -xbar/sqrt(xbar^2 + Sxx/n)
# theta
th = G[(G.B > 0) & (G.A < 0.5)].copy(); th['theta'] = th.B / (0.5 - th.A); th = th[(th.theta > 200) & (th.theta < 800)]
O['n_theta'] = len(th); th.to_csv('out/theta.csv', index=False)
O['theta_extrap'] = dict(inside=int(((th.theta >= th.Tmin) & (th.theta <= th.Tmax)).sum()), outside=int(((th.theta < th.Tmin) | (th.theta > th.Tmax)).sum()))
# A distribution
O['A_median_G'] = float(G.A.median()); O['A_IQR_G'] = [float(G.A.quantile(.25)), float(G.A.quantile(.75))]
O['n_multi_source_pairs'] = int((F.nref > 1).sum()); O['n_multi_source_G'] = int((G.nref > 1).sum())
O['dT_median_G'] = float(G.dT.median()); O['n_le3_G'] = int((G.n == 3).sum())
O['seB_median_G'] = float(G.se_B.median()); O['B_sd_G'] = float(G.B.std())
json.dump(O, open('out/fit.json', 'w'), indent=1, default=float)
print(json.dumps(O, indent=1, default=float, ensure_ascii=False))

{
 "n_pairs_multiT": 143,
 "n_R2_07": 65,
 "n_R2_09": 30,
 "R2_median": 0.625320811884103,
 "repro": {
  "slope": -0.0025587258948823748,
  "intercept": 1.4718299197325069,
  "r": -0.9590879677584845,
  "p": 3.0982389210698073e-36,
  "Tcomp": 390.8195098193471
 },
 "frac_Bpos": 0.8461538461538461,
 "n_polymers_G": 14,
 "Thm_harmonic_of_pairs": 379.89465462669284,
 "Thm_mean": 393.6177767108002,
 "identity_maxdev": 0.0,
 "sd_B_over_Thm": 4.635166896949022,
 "sd_chi_hm": 0.7541262785445523,
 "r_chihm_B": [
  0.3284108566041097,
  0.007565303484832493
 ],
 "rho_chihm_B": [
  0.4745244157234289,
  6.510748376964905e-05
 ],
 "null_r_mean": -0.9653571206978522,
 "null_r_CI": [
  -0.9730743370988056,
  -0.9565956471426277
 ],
 "null_Tcomp_median": 369.9190906473614,
 "null_Tcomp_CI": [
  353.6673816129778,
  384.32494389765367
 ],
 "frac_null_r_le_obs": 0.923,
 "subsets": [
  {
   "label": "R² > 0.7 (as in draft)",
   "n": 65,
   "r": -0.9590879677584845,
   "Tcomp": 390.8195098193471,
   "Th

In [2]:
!pip3 install statsmodels

In [9]:
import os
if os.path.exists('out/fit.json'):
    os.remove('out/fit.json')

In [22]:
import json, os, sys
import numpy as np
import pandas as pd
from scipy import stats

try:
    sys.stdout.reconfigure(encoding='utf-8')
except Exception:
    pass

W = os.path.expanduser('~/Downloads')
IN = f'{W}/chi_parameter.csv'
OUT = f'{W}/out'
os.makedirs(OUT, exist_ok=True)

# load
d = pd.read_csv(IN, encoding='utf-8')
for c in ['Polymer', 'Solvent', 'Polymer_SMILES', 'Solvent_SMILES']:
    d[c] = d[c].astype(str).str.strip()
d['Polymer'] = d.Polymer.str.replace('µ', 'ε')
d['Reference'] = d.Reference.astype(str)
d['src'] = d.Reference.map(
    lambda r: 'Orwoll & Arnold 2007' if '69002-5_14' in r
    else 'Polymer Handbook 4th ed.' if 'Brandrup' in r
    else 'Polymer 1980' if '90035' in r
    else 'other')

# numeric coercion
for c in ['temperature', 'chi']:
    d[c] = pd.to_numeric(d[c].astype(str).str.replace(',', '.'), errors='coerce')
n_before = len(d)
d = d.dropna(subset=['temperature', 'chi']).reset_index(drop=True)
print(f"rows kept: {len(d)} / {n_before}")

O = {}

# 1. per-pair fits
rows = []
for (p, s), g in d.groupby(['Polymer', 'Solvent']):
    if g.temperature.nunique() < 3:
        continue
    x = 1.0 / g.temperature.values
    y = g.chi.values
    n = len(y)
    r = stats.linregress(x, y)
    Thm = n / np.sum(1.0 / g.temperature.values)
    resid = y - (r.intercept + r.slope * x)
    s2 = np.sum(resid ** 2) / (n - 2) if n > 2 else np.nan
    Sxx = np.sum((x - x.mean()) ** 2)
    rows.append(dict(
        polymer=p, solvent=s, n=n, nref=g.Reference.nunique(),
        srcs='; '.join(sorted(g.src.unique())),
        Tmin=g.temperature.min(), Tmax=g.temperature.max(),
        dT=g.temperature.max() - g.temperature.min(),
        Thm=Thm, A=r.intercept, B=r.slope, R2=r.rvalue ** 2,
        se_B=np.sqrt(s2 / Sxx) if (n > 2 and Sxx > 0) else np.nan,
        se_A=np.sqrt(s2 * (1 / n + x.mean() ** 2 / Sxx)) if (n > 2 and Sxx > 0) else np.nan,
        chi_hm=r.intercept + r.slope / Thm,
        chi_mean=y.mean(),
    ))
F = pd.DataFrame(rows)
F.to_csv(f'{OUT}/fits.csv', index=False, encoding='utf-8')
G = F[F.R2 > 0.7].copy()

O['n_pairs_multiT'] = int(len(F))
O['n_R2_07'] = int(len(G))
O['n_R2_09'] = int((F.R2 > 0.9).sum())
O['R2_median'] = float(F.R2.median())

# A on B
f = stats.linregress(G.B, G.A)
O['repro'] = dict(slope=float(f.slope), intercept=float(f.intercept),
                  r=float(f.rvalue), p=float(f.pvalue),
                  Tcomp=float(-1 / f.slope))
O['frac_Bpos'] = float((G.B > 0).mean())
O['n_polymers_G'] = int(G.polymer.nunique())
O['Thm_harmonic_of_pairs'] = float(len(G) / np.sum(1 / G.Thm))
O['Thm_mean'] = float(G.Thm.mean())

# identity check
O['identity_maxdev'] = float(np.max(np.abs(G.A - (G.chi_hm - G.B / G.Thm))))
O['sd_B_over_Thm'] = float((G.B / G.Thm).std())
O['sd_chi_hm'] = float(G.chi_hm.std())
O['r_chihm_B'] = [float(v) for v in stats.pearsonr(G.chi_hm, G.B)]
O['rho_chihm_B'] = [float(v) for v in stats.spearmanr(G.chi_hm, G.B)]

# permutation null
rng = np.random.default_rng(0)
rn, tn = [], []
for _ in range(10000):
    ch = rng.permutation(G.chi_hm.values)
    Ast = ch - G.B.values / G.Thm.values
    q = stats.linregress(G.B.values, Ast)
    rn.append(q.rvalue)
    tn.append(-1 / q.slope)
O['null_r_mean'] = float(np.mean(rn))
O['null_r_CI'] = [float(np.percentile(rn, 2.5)), float(np.percentile(rn, 97.5))]
O['null_Tcomp_median'] = float(np.median(tn))
O['null_Tcomp_CI'] = [float(np.percentile(tn, 2.5)), float(np.percentile(tn, 97.5))]
O['frac_null_r_le_obs'] = float(np.mean(np.array(rn) <= f.rvalue))

# subsets
def sub(X, lab):
    if len(X) < 3:
        return dict(label=lab, n=len(X), r=None, Tcomp=None, Thm=None, r_chihm_B=None)
    q = stats.linregress(X.B, X.A)
    return dict(label=lab, n=int(len(X)), r=float(q.rvalue),
                Tcomp=float(-1 / q.slope),
                Thm=float(len(X) / np.sum(1 / X.Thm)),
                r_chihm_B=float(stats.pearsonr(X.chi_hm, X.B)[0]))

O['subsets'] = [
    sub(G, 'R² > 0.7 (as in draft)'),
    sub(G[G.dT > 50], 'R² > 0.7, ΔT > 50 K'),
    sub(G[G.nref == 1], 'R² > 0.7, single source'),
    sub(F, 'all 143 pairs'),
    sub(F[F.nref == 1], 'all pairs, single source'),
    sub(F[(F.nref == 1) & (F.dT > 50)], 'single source, ΔT > 50 K'),
]

# partial correlation (A, B | Thm)
def pcorr(a, b, c):
    ra = a - np.polyval(np.polyfit(c, a, 1), c)
    rb = b - np.polyval(np.polyfit(c, b, 1), c)
    return float(stats.pearsonr(ra, rb)[0])
O['partial_draft'] = pcorr(G.A.values, G.B.values, G.Thm.values)

# variance of chi(T) across pairs
Ts = np.arange(250, 701, 5)
v = [float(np.var(G.A + G.B / T)) for T in Ts]
O['var_min_T'] = float(Ts[int(np.argmin(v))])
O['data_T_range'] = [float(G.Tmin.min()), float(G.Tmax.max())]

# theta
th = G[(G.B > 0) & (G.A < 0.5)].copy()
th['theta'] = th.B / (0.5 - th.A)
th = th[(th.theta > 200) & (th.theta < 800)].copy()
den = 0.5 - th.A
th['sig_theta'] = np.sqrt((th.se_B / den) ** 2 + (th.B * th.se_A / den ** 2) ** 2)
th['theta_inside_range'] = (th.theta >= th.Tmin) & (th.theta <= th.Tmax)
O['n_theta'] = int(len(th))
O['theta_err'] = dict(median_rel=float((th.sig_theta / th.theta).median()),
                      frac_rel_gt_50=float((th.sig_theta / th.theta > 0.5).mean()),
                      frac_rel_gt_100=float((th.sig_theta / th.theta > 1.0).mean()))
th.to_csv(f'{OUT}/theta.csv', index=False, encoding='utf-8')
O['theta_extrap'] = dict(
    inside=int(((th.theta >= th.Tmin) & (th.theta <= th.Tmax)).sum()),
    outside=int(((th.theta < th.Tmin) | (th.theta > th.Tmax)).sum()))

O['A_median_G'] = float(G.A.median())
O['A_IQR_G'] = [float(G.A.quantile(.25)), float(G.A.quantile(.75))]
O['n_multi_source_pairs'] = int((F.nref > 1).sum())
O['n_multi_source_G'] = int((G.nref > 1).sum())
O['dT_median_G'] = float(G.dT.median())
O['n_le3_G'] = int((G.n == 3).sum())
O['seB_median_G'] = float(G.se_B.median())
O['B_sd_G'] = float(G.B.std())

# 2. within-fit correlation, conditioning, reliable subset
cw = []
for (p, s), g in d.groupby(['Polymer', 'Solvent']):
    if g.temperature.nunique() < 3:
        continue
    x = 1.0 / g.temperature.values
    n = len(x)
    Sxx = np.sum((x - x.mean()) ** 2)
    cw.append(-x.mean() / np.sqrt(x.mean() ** 2 + Sxx / n))
O['within_fit_corr_median'] = float(np.median(cw))
O['within_fit_corr_max'] = float(np.max(cw))

O['seA_median_G'] = float(G.se_A.median())
O['frac_A_not_sig_G'] = float((np.abs(G.A) < 2 * G.se_A).mean())
O['extrap_ratio_median'] = float(np.median((1 / G.Tmin - 1 / G.Tmax) / (1 / G.Tmax)))
G['chiH'] = G.B / G.Thm
O['chiH_frac_median'] = float(np.median(G.chiH / G.chi_hm))

Rl = F[(F.n >= 4) & (F.nref == 1) & (F.dT >= 40) &
       (F.se_B < 0.3 * F.B.abs())].copy()
Rl.to_csv(f'{OUT}/reliable.csv', index=False, encoding='utf-8')

O['reliable'] = dict(
    n=int(len(Rl)),
    polymers=int(Rl.polymer.nunique()),
    r_AB=float(stats.pearsonr(Rl.A, Rl.B)[0]) if len(Rl) > 3 else None,
    Tcomp=float(-1 / stats.linregress(Rl.B, Rl.A).slope) if len(Rl) > 3 else None,
    Thm=float(len(Rl) / np.sum(1 / Rl.Thm)),
    r_chihm_B=float(stats.pearsonr(Rl.chi_hm, Rl.B)[0]),
    p_chihm_B=float(stats.pearsonr(Rl.chi_hm, Rl.B)[1]),
    A_median=float(Rl.A.median()),
    A_IQR=[float(Rl.A.quantile(.25)), float(Rl.A.quantile(.75))],
    frac_Bpos=float((Rl.B > 0).mean()),
    chiH_frac_median=float(np.median((Rl.B / Rl.Thm) / Rl.chi_hm)),
)
rng1 = np.random.default_rng(1)
rn1 = []
for _ in range(5000):
    ch = rng1.permutation(Rl.chi_hm.values)
    rn1.append(stats.pearsonr(Rl.B.values, ch - Rl.B.values / Rl.Thm.values)[0])
O['reliable']['null_r_CI'] = [float(np.percentile(rn1, 2.5)),
                              float(np.percentile(rn1, 97.5))]
O['reliable']['null_r_mean'] = float(np.mean(rn1))
O['reliable']['frac_null_r_le_obs'] = (
    float(np.mean(np.array(rn1) <= O['reliable']['r_AB']))
    if O['reliable']['r_AB'] is not None else None)

# Revision blocks

# residual compensation (collinear with 1/Thm) 
G['invThm'] = 1.0 / G.Thm
a_res = G.A - np.polyval(np.polyfit(G.invThm, G.A, 1), G.invThm)
b_res = G.B - np.polyval(np.polyfit(G.invThm, G.B, 1), G.invThm)
q = stats.pearsonr(a_res, b_res)
O['resid_comp'] = dict(
    r=float(q[0]), p=float(q[1]),
    slope=float(np.polyfit(b_res, a_res, 1)[0]),
)
O['resid_comp']['note'] = ('collinear with 1/Thm; '
                           'see pcorr_chi_hm_B_given_Thm for the interpretable test')

# partial correlations
def pcorr_xy(a, b, c):
    ra = a - np.polyval(np.polyfit(c, a, 1), c)
    rb = b - np.polyval(np.polyfit(c, b, 1), c)
    return stats.pearsonr(ra, rb)
O['pcorr_chi_hm_B_given_Thm'] = list(pcorr_xy(G.chi_hm.values, G.B.values, G.Thm.values))
O['pcorr_chi_hm_Thm_given_B'] = list(pcorr_xy(G.chi_hm.values, G.Thm.values, G.B.values))
O['r_chi_hm_Thm'] = list(stats.pearsonr(G.chi_hm, G.Thm))

# Thm-stratified permutation null 
rng_bin = np.random.default_rng(42)
rn_bin = []
bins = pd.qcut(G.Thm, 4, labels=False, duplicates='drop')
for _ in range(10000):
    ch = G.chi_hm.values.copy()
    for b in np.unique(bins):
        idx = np.where(bins.values == b)[0]
        ch[idx] = rng_bin.permutation(ch[idx])
    rn_bin.append(stats.pearsonr(G.B.values, ch - G.B.values / G.Thm.values)[0])
O['null_r_binned_CI'] = [float(np.percentile(rn_bin, 2.5)),
                         float(np.percentile(rn_bin, 97.5))]
O['null_r_binned_mean'] = float(np.mean(rn_bin))

# robust chi_hm-B + bootstrap CI 
G2 = G[G.B.abs() < 4000]
O['r_chihm_B_robust'] = list(stats.pearsonr(G2.chi_hm, G2.B))
O['n_robust'] = int(len(G2))
rng_bs = np.random.default_rng(7)
bs = []
for _ in range(5000):
    i = rng_bs.integers(0, len(G), len(G))
    bs.append(stats.pearsonr(G.chi_hm.values[i], G.B.values[i])[0])
O['r_chihm_B_boot_CI'] = [float(np.percentile(bs, 2.5)),
                          float(np.percentile(bs, 97.5))]

# IGC range + B>0 & A<0 
igc = d[d.Reference.str.contains('90035', na=False)]
igc_hi = igc[igc.temperature > 400]
O['IGC_range'] = [float(igc_hi.temperature.min()), float(igc_hi.temperature.max())]
O['frac_Bpos_and_Aneg'] = float(((G.B > 0) & (G.A < 0)).mean())

# PS / cyclohexane 
pc = d[(d.Polymer == 'polystyrene') & (d.Solvent == 'cyclohexane') &
       (d.temperature.between(295, 310))]
O['PS_cy_near_theta'] = dict(
    n=int(len(pc)),
    chi_min=float(pc.chi.min()) if len(pc) else None,
    chi_max=float(pc.chi.max()) if len(pc) else None)
sub_ps = F[(F.polymer == 'polystyrene') & (F.solvent == 'cyclohexane')]
if len(sub_ps):
    ps = sub_ps.iloc[0]
    O['PS_cy_fit'] = dict(R2=float(ps.R2), B=float(ps.B), A=float(ps.A),
                          n=int(ps.n), nref=int(ps.nref))
else:
    O['PS_cy_fit'] = None

# theta list 
O['theta_list'] = th[['polymer', 'solvent', 'Tmin', 'Tmax', 'theta']].round(0).to_dict('records')
O['n_pairs_dT_le5_G'] = int((G.dT <= 5).sum())
O['n_pairs_dT_lt20_G'] = int((G.dT < 20).sum())

O['pairs_mixing_IGC1980_lowT'] = int(sum(
    1 for _, g in d.groupby(['Polymer', 'Solvent'])
    if g.temperature.nunique() >= 3
    and g.src.eq('Polymer 1980').any()
    and (g.temperature < 400).any()))

O['n_points'] = int(len(d))
O['n_pairs'] = int(d.groupby(['Polymer', 'Solvent']).ngroups)
O['src_share'] = d.src.value_counts(normalize=True).round(3).to_dict()
O['n_single_T_pairs'] = int(
    (d.groupby(['Polymer', 'Solvent']).temperature.nunique() == 1).sum())

# save
with open(f'{OUT}/fit.json', 'w', encoding='utf-8') as f:
    json.dump(O, f, indent=1, default=float, ensure_ascii=False)

# safe print
KEYS = ['n_pairs_multiT', 'n_R2_07', 'repro', 'null_r_CI',
        'frac_null_r_le_obs', 'reliable', 'within_fit_corr_median',
        'frac_A_not_sig_G', 'PS_cy_fit',
        'resid_comp', 'pcorr_chi_hm_B_given_Thm', 'pcorr_chi_hm_Thm_given_B',
        'r_chi_hm_Thm', 'null_r_binned_CI', 'null_r_binned_mean',
        'r_chihm_B_robust', 'r_chihm_B_boot_CI', 'n_robust',
        'IGC_range', 'frac_Bpos_and_Aneg',
        'theta_err', 'theta_extrap', 'n_theta']
missing = [k for k in KEYS if k not in O]
if missing:
    print('MISSING keys:', missing)
print(json.dumps({k: O[k] for k in KEYS if k in O},
                 indent=1, default=float, ensure_ascii=True))
print("\nOK: wrote", f'{OUT}/fit.json')

rows kept: 1586 / 1586
{
 "n_pairs_multiT": 143,
 "n_R2_07": 65,
 "repro": {
  "slope": -0.0025587258948823748,
  "intercept": 1.4718299197325069,
  "r": -0.9590879677584845,
  "p": 3.0982389210698073e-36,
  "Tcomp": 390.8195098193471
 },
 "null_r_CI": [
  -0.9730743370988056,
  -0.9565956471426277
 ],
 "frac_null_r_le_obs": 0.923,
 "reliable": {
  "n": 14,
  "polymers": 4,
  "r_AB": -0.9878903205083167,
  "Tcomp": 412.16851598344965,
  "Thm": 439.1086358030844,
  "r_chihm_B": 0.1494648737894373,
  "p_chihm_B": 0.6100611189413461,
  "A_median": -0.24647013173760235,
  "A_IQR": [
   -1.8537208893096393,
   0.18827487710767626
  ],
  "frac_Bpos": 0.7857142857142857,
  "chiH_frac_median": 1.1982399998769349,
  "null_r_CI": [
   -0.9968013164436346,
   -0.9858569052998898
  ],
  "null_r_mean": -0.9907826877017802,
  "frac_null_r_le_obs": 0.8332
 },
 "within_fit_corr_median": -0.9978170381515153,
 "frac_A_not_sig_G": 0.38461538461538464,
 "PS_cy_fit": {
  "R2": 0.0789131701612333,
  "B": -8

In [23]:
a_res = G.A - np.polyval(np.polyfit(G.invThm, G.A, 1), G.invThm)
b_res = G.B - np.polyval(np.polyfit(G.invThm, G.B, 1), G.invThm)
q = stats.pearsonr(a_res, b_res)
print('manual check:', q, np.polyfit(b_res, a_res, 1)[0])

manual check: PearsonRResult(statistic=-0.9667442555956125, pvalue=5.08893911227496e-39) -0.0027935699193022356


In [24]:
import os
print("cwd:", os.getcwd())
print("exists Downloads/chi_parameter.csv:", os.path.exists('Downloads/chi_parameter.csv'))
print("exists ~/Downloads/chi_parameter.csv:", os.path.exists(os.path.expanduser('~/Downloads/chi_parameter.csv')))

cwd: C:\Users\130\Downloads
exists Downloads/chi_parameter.csv: False
exists ~/Downloads/chi_parameter.csv: True


In [25]:
print(d.temperature.dtype)
print(d.temperature.head(20))
print(d.temperature[pd.to_numeric(d.temperature, errors='coerce').isna()].head(20))

float64
0     373.0
1     373.0
2     393.0
3     373.0
4     373.0
5     502.0
6     413.0
7     408.0
8     502.0
9     296.5
10    502.0
11    296.5
12    353.0
13    353.0
14    373.0
15    413.0
16    373.0
17    413.0
18    373.0
19    502.0
Name: temperature, dtype: float64
Series([], Name: temperature, dtype: float64)


In [26]:
"""Addition to the revision: is there a temperature signal at all?

The compensation line follows from the algebra of the fit. A prior question is
whether chi(T) in this compilation carries a temperature dependence large enough
to be estimated. We compare, within each temperature series, the spread
attributable to the temperature with the step between sources measured at
nearly the same temperature, and we report how often the slope B is
distinguishable from zero.
"""
import json, os, sys
import numpy as np
import pandas as pd
from scipy import stats

try:
    sys.stdout.reconfigure(encoding='utf-8')
except Exception:
    pass

W = os.path.expanduser('~/Downloads')
IN = f'{W}/chi_parameter.csv'
OUT = f'{W}/out'
os.makedirs(OUT, exist_ok=True)
O = {}

# load and clean
if not os.path.exists(IN):
    raise FileNotFoundError(f"Not found: {IN}  (cwd = {os.getcwd()})")

d = pd.read_csv(IN, encoding='utf-8')
for c in ['Polymer', 'Solvent', 'Polymer_SMILES', 'Solvent_SMILES']:
    d[c] = d[c].astype(str).str.strip()
d['Polymer'] = d.Polymer.str.replace('µ', 'ε')

# coerce numeric columns; report what was dropped
for c in ['temperature', 'chi']:
    d[c] = pd.to_numeric(d[c].astype(str).str.replace(',', '.'), errors='coerce')
    n_bad = int(d[c].isna().sum())
    if n_bad:
        print(f"WARNING: {c}: {n_bad} non-numeric values -> NaN")
n_before = len(d)
d = d.dropna(subset=['temperature', 'chi']).reset_index(drop=True)
print(f"Rows kept after numeric coercion: {len(d)} / {n_before}")

d['src'] = d.Reference.map(
    lambda r: 'Orwoll & Arnold 2007' if '69002-5_14' in str(r)
    else 'Polymer Handbook 4th ed.' if 'Brandrup' in str(r)
    else 'Polymer 1980' if '90035' in str(r)
    else 'other')

# helpers
def fit_series(a):
    """Fit chi = A + B/T to a series of (temperature, chi) points.

    Returns a dict or None if fewer than 3 points. `a` must have columns
    `temperature` and `chi`.
    """
    x = 1.0 / a.temperature.values
    y = a.chi.values
    n = len(y)
    if n < 3:
        return None
    r = stats.linregress(x, y)
    res = y - (r.intercept + r.slope * x)
    dof = n - 2
    s2 = np.sum(res ** 2) / dof if dof > 0 else np.nan
    Sxx = np.sum((x - x.mean()) ** 2)
    seB = np.sqrt(s2 / Sxx) if (Sxx > 0 and np.isfinite(s2)) else np.nan
    dif = np.diff(y)
    tol = 1e-9
    return dict(
        n=n,
        dT=float(a.temperature.max() - a.temperature.min()),
        rng=float(y.max() - y.min()),
        B=float(r.slope),
        seB=float(seB) if np.isfinite(seB) else np.nan,
        tB=float(abs(r.slope) / seB) if (np.isfinite(seB) and seB > 0) else np.nan,
        R2=float(r.rvalue ** 2),
        monotone=bool(np.all(dif >= -tol) or np.all(dif <= tol)),
    )


def steps_between_sources(dTmax=10.0, bin_width=2.0):
    """Differences between records from different sources at nearly the same T.

    Points are first averaged within (source, T-bin) so that repeated
    measurements from the same source do not inflate the count of steps.
    NOTE: the aggregated temperature column is named `Tmid`, not `T`, because
    `gb.T` is the DataFrame transpose attribute and would silently break.
    """
    out = []
    for (p, s), g in d.groupby(['Polymer', 'Solvent']):
        if g.src.nunique() < 2:
            continue
        g = g.copy()
        g['Tbin'] = (g.temperature / bin_width).round() * bin_width
        gb = (g.groupby(['src', 'Tbin'], as_index=False)
                .agg(Tmid=('temperature', 'mean'), chi=('chi', 'mean')))
        gb = gb.dropna(subset=['Tmid', 'chi']).reset_index(drop=True)
        for i in range(len(gb)):
            for j in range(i + 1, len(gb)):
                if gb.src.iloc[i] == gb.src.iloc[j]:
                    continue
                if abs(gb.Tmid.iloc[i] - gb.Tmid.iloc[j]) > dTmax:
                    continue
                out.append(dict(
                    polymer=p, solvent=s,
                    src_i=gb.src.iloc[i], src_j=gb.src.iloc[j],
                    T=0.5 * (gb.Tmid.iloc[i] + gb.Tmid.iloc[j]),
                    dT=abs(gb.Tmid.iloc[i] - gb.Tmid.iloc[j]),
                    dchi=abs(gb.chi.iloc[i] - gb.chi.iloc[j]),
                    chi=0.5 * (gb.chi.iloc[i] + gb.chi.iloc[j]),
                ))
    return pd.DataFrame(out)


def steps_within_source(dTmax=10.0):
    """Consecutive differences within a single source at nearly the same T.

    Control: if within-source scatter is comparable to between-source scatter,
    the compilation itself is not the main source of noise.
    """
    out = []
    for (p, s), g in d.groupby(['Polymer', 'Solvent']):
        for src, gg in g.groupby('src'):
            gg = gg.sort_values('temperature').reset_index(drop=True)
            for i in range(len(gg) - 1):
                if abs(gg.temperature.iloc[i + 1] - gg.temperature.iloc[i]) > dTmax:
                    continue
                out.append(abs(gg.chi.iloc[i + 1] - gg.chi.iloc[i]))
    return np.array(out)


def frac_tB_lt2(S, mask=None, label='all'):
    X = S if mask is None else S[mask]
    if 'tB' not in X or len(X) == 0:
        return dict(label=label, n=0, frac=None)
    t = pd.to_numeric(X.tB, errors='coerce')
    ok = t.notna()
    X = X[ok]; t = t[ok]
    if len(X) == 0:
        return dict(label=label, n=0, frac=None)
    return dict(label=label, n=int(len(X)), frac=float((t < 2).mean()))


# 1. shape of chi(T)
#    S_avg: fit on values averaged over sources at each T (shape of chi(T))
#    S_raw: fit on all points (spread of the compilation, not only the mean)
rows_avg, rows_raw = [], []
for (p, s), g in d.groupby(['Polymer', 'Solvent']):
    if g.temperature.nunique() < 3:
        continue
    nref = int(g.Reference.nunique())

    a = (g.groupby('temperature', as_index=False).chi.mean()
          .sort_values('temperature'))
    fa = fit_series(a)
    if fa is not None:
        fa.update(polymer=p, solvent=s, nref=nref)
        rows_avg.append(fa)

    gr = g.sort_values('temperature')[['temperature', 'chi']]
    fr = fit_series(gr)
    if fr is not None:
        fr.update(polymer=p, solvent=s, nref=nref)
        rows_raw.append(fr)

S_avg = pd.DataFrame(rows_avg)
S_raw = pd.DataFrame(rows_raw)
S_avg.to_csv(f'{OUT}/tsignal_series_avg.csv', index=False, encoding='utf-8')
S_raw.to_csv(f'{OUT}/tsignal_series_raw.csv', index=False, encoding='utf-8')

if len(S_avg) == 0:
    raise RuntimeError("S_avg is empty: no series with >= 3 distinct temperatures")

O['series'] = dict(
    n_avg=int(len(S_avg)),
    n_raw=int(len(S_raw)),
    R2_median_avg=float(S_avg.R2.median()),
    R2_median_raw=float(S_raw.R2.median()),
    frac_R2_below_09_avg=float((S_avg.R2 < 0.9).mean()),
    frac_R2_below_07_avg=float((S_avg.R2 < 0.7).mean()),
    n_nonmonotone_avg=int((~S_avg.monotone).sum()),
    frac_nonmonotone_avg=float((~S_avg.monotone).mean()),
    range_median_avg=float(S_avg.rng.median()),
    range_median_raw=float(S_raw.rng.median()) if len(S_raw) else None,
    dT_median_avg=float(S_avg.dT.median()),
    frac_range_below_01_avg=float((S_avg.rng < 0.1).mean()),
    tB_insignificant=dict(
        all=frac_tB_lt2(S_avg, label='all'),
        n3=frac_tB_lt2(S_avg, S_avg.n == 3, label='n = 3'),
        n4_5=frac_tB_lt2(S_avg, S_avg.n.between(4, 5), label='n = 4-5'),
        n6plus=frac_tB_lt2(S_avg, S_avg.n >= 6, label='n >= 6'),
        R2_07=frac_tB_lt2(S_avg, S_avg.R2 > 0.7, label='R2 > 0.7'),
    ),
)

# 2. step between sources at nearly the same temperature
ST = steps_between_sources(dTmax=10.0, bin_width=2.0)
ST.to_csv(f'{OUT}/tsignal_source_steps.csv', index=False, encoding='utf-8')

if len(ST):
    O['source_step'] = dict(
        n=int(len(ST)),
        n_pairs=int(ST.groupby(['polymer', 'solvent']).ngroups),
        median=float(ST.dchi.median()),
        q90=float(ST.dchi.quantile(.9)),
        maximum=float(ST.dchi.max()),
        frac_above_01=float((ST.dchi > 0.1).mean()),
        src_composition=ST.apply(
            lambda r: ' & '.join(sorted([r.src_i, r.src_j])), axis=1
        ).value_counts().to_dict(),
    )
else:
    O['source_step'] = dict(n=0, n_pairs=0, median=None, q90=None,
                            maximum=None, frac_above_01=None,
                            src_composition={})

WI = steps_within_source(dTmax=10.0)
O['within_source_step'] = dict(
    n=int(len(WI)),
    median=float(np.median(WI)) if len(WI) else None,
    q90=float(np.quantile(WI, .9)) if len(WI) else None,
)

# 3. the comparison that matters
#    Use S_raw (not S_avg) so that inter-source spread is not averaged away.
if len(ST) and len(S_raw):
    med_step = float(ST.dchi.median())
    O['comparison'] = dict(
        source_step_median=med_step,
        source_step_q90=float(ST.dchi.quantile(.9)),
        frac_series_range_below_q90_step=float((S_raw.rng < ST.dchi.quantile(.9)).mean()),
        frac_series_range_below_2x_median_step=float((S_raw.rng < 2 * med_step).mean()),
        ratio_of_medians=float(S_raw.rng.median() / med_step) if med_step > 0 else None,
        median_of_ratios=float(np.median(S_raw.rng / med_step)) if med_step > 0 else None,
    )
else:
    O['comparison'] = None

# save
with open(f'{OUT}/tsignal.json', 'w', encoding='utf-8') as f:
    json.dump(O, f, indent=1, default=float, ensure_ascii=False)

print(json.dumps(O, indent=1, default=float, ensure_ascii=True))

Rows kept after numeric coercion: 1586 / 1586
{
 "series": {
  "n_avg": 143,
  "n_raw": 143,
  "R2_median_avg": 0.625320811884103,
  "R2_median_raw": 0.625320811884103,
  "frac_R2_below_09_avg": 0.7902097902097902,
  "frac_R2_below_07_avg": 0.5454545454545454,
  "n_nonmonotone_avg": 105,
  "frac_nonmonotone_avg": 0.7342657342657343,
  "range_median_avg": 0.22699999999999998,
  "range_median_raw": 0.22699999999999998,
  "dT_median_avg": 68.0,
  "frac_range_below_01_avg": 0.26573426573426573,
  "tB_insignificant": {
   "all": {
    "label": "all",
    "n": 143,
    "frac": 0.5104895104895105
   },
   "n3": {
    "label": "n = 3",
    "n": 32,
    "frac": 0.625
   },
   "n4_5": {
    "label": "n = 4-5",
    "n": 78,
    "frac": 0.358974358974359
   },
   "n6plus": {
    "label": "n >= 6",
    "n": 33,
    "frac": 0.7575757575757576
   },
   "R2_07": {
    "label": "R2 > 0.7",
    "n": 65,
    "frac": 0.06153846153846154
   }
  }
 },
 "source_step": {
  "n": 109,
  "n_pairs": 69,
  "median

In [27]:
import json
with open('out/fit.json', encoding='utf-8') as f:
    lines = f.readlines()
print("total lines:", len(lines))
for i in range(max(0, 36), min(len(lines), 46)):
    print(f"{i+1:3d}: {repr(lines[i])}")

total lines: 480
 37: ' ],\n'
 38: ' "frac_null_r_le_obs": 0.923,\n'
 39: ' "subsets": [\n'
 40: '  {\n'
 41: '   "label": "R² > 0.7 (as in draft)",\n'
 42: '   "n": 65,\n'
 43: '   "r": -0.9590879677584845,\n'
 44: '   "Tcomp": 390.8195098193471,\n'
 45: '   "Thm": 379.89465462669284,\n'
 46: '   "r_chihm_B": 0.3284108566041097\n'


In [30]:
"""Figure for the new subsection: temperature signal against scatter at fixed T.

Panels
(a) ECDF of three quantities on a common scale:
      - range of chi within a series (from tsignal_series_raw.csv);
      - |dchi| at nearly the same T, within one source;
      - |dchi| at nearly the same T, between different sources.
    Plotted as ECDF, not stacked histograms, because the three samples
    differ in size and in meaning.
(b) Distribution of |B|/sigma(B) for R^2 <= 0.7 and R^2 > 0.7, normalised
    to unit area (density=True), with the 2-sigma threshold marked.
(c) R^2 vs measured range dT, coloured by monotonicity, threshold R^2 = 0.7.
"""
import os, json, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D

try:
    sys.stdout.reconfigure(encoding='utf-8')
except Exception:
    pass

# paths: robust in Jupyter and in scripts
W = os.environ.get('CHI_ROOT', os.path.expanduser('~/Downloads'))
OUT = os.path.join(W, 'out')
FIG = os.path.join(OUT, 'en')
os.makedirs(FIG, exist_ok=True)

IN_CSV     = os.path.join(W, 'chi_parameter.csv')
S_RAW_CSV  = os.path.join(OUT, 'tsignal_series_raw.csv')
S_AVG_CSV  = os.path.join(OUT, 'tsignal_series_avg.csv')
ST_CSV     = os.path.join(OUT, 'tsignal_source_steps.csv')
TS_JSON    = os.path.join(OUT, 'tsignal.json')

for p in (IN_CSV, S_RAW_CSV, S_AVG_CSV, ST_CSV, TS_JSON):
    if not os.path.exists(p):
        raise FileNotFoundError(p)

S_raw = pd.read_csv(S_RAW_CSV, encoding='utf-8')
S_avg = pd.read_csv(S_AVG_CSV, encoding='utf-8')
ST    = pd.read_csv(ST_CSV, encoding='utf-8')
with open(TS_JSON, encoding='utf-8') as f:
    O = json.load(f)

d = pd.read_csv(IN_CSV, encoding='utf-8')
for c in ['Polymer', 'Solvent', 'Reference']:
    d[c] = d[c].astype(str).str.strip()
d['temperature'] = pd.to_numeric(d.temperature.astype(str).str.replace(',', '.'),
                                 errors='coerce')
d['chi']         = pd.to_numeric(d.chi.astype(str).str.replace(',', '.'),
                                 errors='coerce')
d = d.dropna(subset=['temperature', 'chi']).reset_index(drop=True)
d['src'] = d.Reference.map(
    lambda r: 'Orwoll' if '69002-5_14' in r
    else 'Handbook' if 'Brandrup' in r
    else 'Gunduz' if '90035' in r
    else 'other')

S_avg['monotone'] = S_avg.monotone.astype(bool)
S_raw['monotone'] = S_raw.monotone.astype(bool)

# within-source steps at nearly the same T.
# Average within (source, 2 K bin), sort by Tmid, then take consecutive pairs
# so repeated points do not inflate n.  NOTE: the aggregated temperature
# column is called Tmid, not T, because gg.T is the DataFrame transpose
# attribute and would raise "truth value of a Series is ambiguous".
def within_source_steps(df, dTmax=10.0, bin_width=2.0):
    out = []
    for (p, s), g in df.groupby(['Polymer', 'Solvent']):
        g = g.copy()
        g['Tbin'] = (g.temperature / bin_width).round() * bin_width
        for src, gg in g.groupby('src'):
            gg = (gg.groupby('Tbin', as_index=False)
                    .agg(Tmid=('temperature', 'mean'), chi=('chi', 'mean'))
                    .sort_values('Tmid').reset_index(drop=True))
            for i in range(len(gg) - 1):
                if abs(gg.Tmid.iloc[i + 1] - gg.Tmid.iloc[i]) > dTmax:
                    continue
                out.append(abs(gg.chi.iloc[i + 1] - gg.chi.iloc[i]))
    return np.array(out)

wi = within_source_steps(d)
print(f'within-source steps: n = {len(wi)}')

# style
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 8,
    'axes.labelsize': 8,
    'axes.titlesize': 9,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 6.8,
    'axes.linewidth': 0.6,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.edgecolor': 'black',
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.size': 3,
    'ytick.major.size': 3,
    'axes.grid': True,
    'grid.color': '0.88',
    'grid.linewidth': 0.4,
    'axes.axisbelow': True,
    'legend.frameon': False,
    'savefig.dpi': 600,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.02,
    'pdf.fonttype': 42,
    'svg.fonttype': 'none',
})

BLUE, ORANGE, GREEN = '#0072B2', '#D55E00', '#009E73'
GRAY, INK = '#999999', '#000000'
FF = FuncFormatter(lambda v, p: ('%g' % v).replace('-', '\u2212'))

def style_ax(ax, panel=None):
    ax.xaxis.set_major_formatter(FF)
    ax.yaxis.set_major_formatter(FF)
    ax.tick_params(which='both', top=False, right=False)
    if panel is not None:
        ax.text(0.02, 0.98, f'({panel})', transform=ax.transAxes,
                ha='left', va='top', fontsize=9, fontweight='bold')

def ecdf(x):
    x = np.sort(np.asarray(x, dtype=float))
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return x, x
    return x, np.arange(1, len(x) + 1) / len(x)

# figure
fig, axs = plt.subplots(1, 3, figsize=(7.1, 2.9))

# (a) ECDF
ax = axs[0]
datasets = [
    (r'range of $\chi$ in a series',           S_raw.rng.values, BLUE),
    (r'$|\Delta\chi|$, same $T$, one source',  wi,               ORANGE),
    (r'$|\Delta\chi|$, same $T$, two sources', ST.dchi.values,   INK),
]
for lab, vals, col in datasets:
    x, y = ecdf(vals)
    if len(x) == 0:
        continue
    ax.step(x, y, where='post', color=col, lw=1.2, label=lab)
ax.set_xlim(0, 1.2)
ax.set_ylim(0, 1.02)
ax.set_xlabel(r'$|\Delta\chi|$ or range of $\chi$')
ax.set_ylabel('cumulative fraction')
ax.set_title('(a) signal and scatter', loc='left', fontsize=9, pad=6)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.30),
          ncol=1, frameon=False,
          handlelength=1.0, handletextpad=0.3, borderpad=0.2)
style_ax(ax)

# (b) |B|/sigma(B)
ax = axs[1]
lo = pd.to_numeric(S_avg.loc[S_avg.R2 <= 0.7, 'tB'], errors='coerce').dropna().values
hi = pd.to_numeric(S_avg.loc[S_avg.R2 > 0.7, 'tB'], errors='coerce').dropna().values
bins = np.linspace(0, 12, 25)
for vals, col, lab in [(lo, GRAY, r'$R^2 \leq 0.7$'),
                       (hi, GREEN, r'$R^2 > 0.7$')]:
    if len(vals) == 0:
        continue
    ax.hist(np.clip(vals, 0, 12), bins=bins, density=True,
            color=col, alpha=0.55, edgecolor='white', lw=0.3,
            label=f'{lab} (n = {len(vals)})')
ax.axvline(2, color=ORANGE, lw=1.2)
# label above the histogram, with a thin leader down to the line
ax.annotate(r'$|B|=2\sigma(B)$', xy=(2.0, 0.85), xycoords=('data', 'axes fraction'),
            xytext=(6, 0), textcoords='offset points',
            rotation=90, va='center', ha='left',
            color=ORANGE, fontsize=7)
# overflow marker on the last bin
y_top = ax.get_ylim()[1]
ax.text(12, y_top * 0.85, r'$>12$', ha='center', va='bottom',
        fontsize=6.5, color=GREEN)
ax.set_xlabel(r'$|B|/\sigma(B)$')
ax.set_ylabel('probability density')
ax.set_title('(b) significance of the slope', loc='left', fontsize=9, pad=6)
ax.legend(loc='upper right', frameon=False,
          handlelength=1.0, handletextpad=0.3, borderpad=0.2)
style_ax(ax)

# (c) R^2 vs dT
ax = axs[2]
for mask, col, lab in [(S_avg.monotone, BLUE, 'monotonic'),
                       (~S_avg.monotone, ORANGE, 'non-monotonic')]:
    X = S_avg.loc[mask, ['dT', 'R2']].dropna()
    if len(X) == 0:
        continue
    ax.scatter(X.dT, X.R2, s=10, color=col, edgecolors='white',
               linewidths=0.2, label=lab, zorder=3)

ax.axhline(0.7, color=INK, lw=0.8, ls='--')

# R^2 = 0.7 label: right side, above the dashed line, clear of points
ax.text(0.97, 0.86, r'$R^2 = 0.7$', transform=ax.transAxes,
        ha='right', va='bottom', fontsize=7, color=INK)

ax.set_xlabel(r'measured range $\Delta T$ (K)')
ax.set_ylabel(r'$R^2$ of $\chi = A + B/T$')
ax.set_ylim(0, 1.02)
ax.set_title('(c) quality of the fit', loc='left', fontsize=9, pad=6)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.30),
          ncol=2, frameon=False,
          handlelength=1.0, handletextpad=0.3, borderpad=0.2,
          columnspacing=1.0)
style_ax(ax)

fig.subplots_adjust(wspace=0.45, bottom=0.36)
for ext in ('pdf', 'svg', 'png'):
    fig.savefig(os.path.join(FIG, f'fig_signal2.{ext}'))
plt.close(fig)

written = ', '.join(os.path.join(FIG, f'fig_signal2.{ext}')
                    for ext in ('pdf', 'svg', 'png'))
print('written', written)

within-source steps: n = 174
written C:\Users\130/Downloads\out\en\fig_signal2.pdf, C:\Users\130/Downloads\out\en\fig_signal2.svg, C:\Users\130/Downloads\out\en\fig_signal2.png


In [34]:
import json, os, sys
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, ScalarFormatter
from matplotlib.lines import Line2D
from scipy import stats

try:
    sys.stdout.reconfigure(encoding='utf-8')
except Exception:
    pass

# load (UTF-8)
with open('out/fit.json', encoding='utf-8') as f:
    O = json.load(f)
d = pd.read_csv('chi_parameter.csv', encoding='utf-8')
for c in ['Polymer', 'Solvent']:
    d[c] = d[c].astype(str).str.strip()
F = pd.read_csv('out/fits.csv', encoding='utf-8')
G = F[F.R2 > 0.7]

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 9,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#52514e', 'xtick.color': '#52514e', 'ytick.color': '#52514e',
    'axes.grid': True, 'grid.color': '#e6e5e1', 'grid.linewidth': 0.6,
    'axes.axisbelow': True, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'legend.frameon': False,
})
BLUE, ORANGE, AQUA, INK, INK2, MUTED, GRAY = '#2a78d6', '#eb6834', '#1baf7a', '#0b0b0b', '#52514e', '#8a8986', '#b9b8b2'

FF = FuncFormatter(lambda v, p: ('%g' % v).replace('.', ',').replace('-', '−'))

def fix(fig):
    for a in fig.axes:
        if isinstance(a.xaxis.get_major_formatter(), ScalarFormatter):
            a.xaxis.set_major_formatter(FF)
        if isinstance(a.yaxis.get_major_formatter(), ScalarFormatter):
            a.yaxis.set_major_formatter(FF)

cm = lambda x, n=2: f'{x:.{n}f}'.replace('.', ',').replace('-', '−')

SRC = lambda r: ('Orwoll & Arnold (2007)' if '69002' in r
                 else 'Polymer Handbook' if 'Brandrup' in r
                 else 'Gunduz & Dincer (1980), IGC' if '90035' in r
                 else 'other')
SC = {'Orwoll & Arnold (2007)': BLUE, 'Polymer Handbook': ORANGE,
      'Gunduz & Dincer (1980), IGC': AQUA, 'other': GRAY}

# Fig 1: examples of pairs with mixed sources
fig, axs = plt.subplots(1, 3, figsize=(7.4, 2.7), sharey=False)
for ax, (p, s, lab) in zip(axs, [('polystyrene', 'cyclohexane', 'PS / cyclohexane'),
                                 ('polystyrene', 'toluene', 'PS / toluene'),
                                 ('polystyrene', 'methanol', 'PS / methanol')]):
    g = d[(d.Polymer == p) & (d.Solvent == s)]
    for src, gg in g.groupby(g.Reference.map(SRC)):
        ax.scatter(gg.temperature, gg.chi, s=22, c=SC[src],
                   edgecolors='white', linewidths=0.7, label=src, zorder=3)
    f = F[(F.polymer == p) & (F.solvent == s)].iloc[0]
    T = np.linspace(g.temperature.min(), g.temperature.max(), 50)
    ax.plot(T, f.A + f.B / T, color=INK, lw=1.2)
    ax.set_title(f'{lab}\nR² = {cm(f.R2)}, n = {int(f.n)}', loc='left', fontsize=8.5)
    ax.set_xlabel('T, K')
    if p == 'polystyrene' and s == 'cyclohexane':
        ax.axhline(0.5, color=MUTED, ls=':', lw=0.9)
        ax.axvline(307.6, color=MUTED, ls='--', lw=0.9)
axs[0].set_ylabel(r'$\chi$')
h, l = [], []
for a in axs:
    for hh, ll in zip(*a.get_legend_handles_labels()):
        if ll not in l:
            h.append(hh); l.append(ll)
fig.legend(h, l, loc='lower center', ncol=4, fontsize=7.5, bbox_to_anchor=(0.5, -0.16))
fix(fig)
fig.savefig('out/fig1_pairs.png'); plt.close(fig)

# Fig 3: artefact
fig, axs = plt.subplots(1, 3, figsize=(7.4, 2.6))

ax = axs[0]
ax.scatter(G.B, G.A, s=18, c=BLUE, edgecolors='white', linewidths=0.6, zorder=3)
xs = np.linspace(G.B.min(), G.B.max(), 10)
r = O['repro']
ax.plot(xs, r['intercept'] + r['slope'] * xs, color=INK, lw=1.2)
ax.set_xlabel('B, K'); ax.set_ylabel('A')
ax.set_title(f"(a) r = {cm(r['r'])}, T$_{{comp}}$ = {r['Tcomp']:.0f} K", loc='left', fontsize=8.5)

ax = axs[1]
rng = np.random.default_rng(0); rn = []
for _ in range(10000):
    ch = rng.permutation(G.chi_hm.values)
    rn.append(stats.pearsonr(G.B.values, ch - G.B.values / G.Thm.values)[0])
ax.hist(rn, bins=40, color=GRAY, edgecolor='white', lw=0.4)
ax.axvline(r['r'], color=ORANGE, lw=1.8)
ax.text(r['r'] - 0.0007, ax.get_ylim()[1] * 0.92, 'observed', fontsize=7, color=INK, ha='right')
ax.set_xlabel(r'r(A, B) with shuffled $\chi(T_{hm})$'); ax.set_ylabel('count')
ax.grid(axis='x', visible=False)
ax.set_title('(b) null model', loc='left', fontsize=8.5)

ax = axs[2]
ax.scatter(G.B, G.chi_hm, s=18, c=AQUA, edgecolors='white', linewidths=0.6, zorder=3)
ax.set_xlabel('B, K'); ax.set_ylabel(r'$\chi(T_{hm})$')
q = stats.pearsonr(G.chi_hm, G.B)
ax.set_title(f'(c) r = {cm(q[0])}, p = {cm(q[1], 3)}', loc='left', fontsize=8.5)

fix(fig); fig.savefig('out/fig3_artefact.png'); plt.close(fig)

# Fig 4: Tcomp vs Thm for subsets
S = pd.DataFrame(O['subsets'])
fig, ax = plt.subplots(figsize=(4.6, 3.2))
ax.scatter(S.Thm, S.Tcomp, s=34, c=BLUE, edgecolors='white', zorder=3)
lab = {'R² > 0.7 (as in draft)': 'R² > 0.7',
       'R² > 0.7, ΔT > 50 K': 'R² > 0.7, ΔT > 50 K',
       'R² > 0.7, single source': 'R² > 0.7, single source',
       'all 143 pairs': 'all 143 pairs',
       'all pairs, single source': 'all, single source',
       'single source, ΔT > 50 K': 'single source, ΔT > 50 K'}
OFF = {'R² > 0.7 (as in draft)': (5, -9),
       'R² > 0.7, single source': (5, 3),
       'all 143 pairs': (-4, 6)}
for _, rr in S.iterrows():
    ax.annotate(lab[rr.label], (rr.Thm, rr.Tcomp), fontsize=6.8,
                xytext=OFF.get(rr.label, (5, 2)), textcoords='offset points',
                color=INK2, ha='right' if rr.label == 'all 143 pairs' else 'left')
rl = O['reliable']
ax.scatter([rl['Thm']], [rl['Tcomp']], s=34, c=ORANGE, edgecolors='white', zorder=3)
ax.annotate('reliable fits (n = %d)' % rl['n'], (rl['Thm'], rl['Tcomp']),
            fontsize=6.8, xytext=(4, -9), textcoords='offset points')
lim = [340, 480]
ax.plot(lim, lim, color=MUTED, ls='--', lw=0.9)
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel(r'Harmonic mean temperature of the measurements, $T_{hm}$ (K)')
ax.set_ylabel(r'$T_{comp}$ from regression A on B (K)')
fix(fig); fig.savefig('out/fig4_Tcomp.png'); plt.close(fig)

# Fig 5: conditioning: se_A vs |A|, within-fit corr
fig, axs = plt.subplots(1, 2, figsize=(7.0, 2.8))
ax = axs[0]
ax.scatter(G.A.abs(), G.se_A, s=16, c=BLUE, edgecolors='white', linewidths=0.5, zorder=3)
m = max(G.A.abs().max(), G.se_A.max())
ax.plot([0, m], [0, m / 2], color=MUTED, ls='--', lw=0.9)
ax.text(m * 0.55, m * 0.31, '|A| = 2σ(A)', fontsize=7, color=INK2, rotation=0)
ax.set_xscale('symlog', linthresh=0.1); ax.set_yscale('symlog', linthresh=0.1)
ax.set_xlabel('|A|'); ax.set_ylabel('σ(A)')
ax.set_title('(a)', loc='left', fontsize=8.5)

ax = axs[1]
x = (1 / G.Tmin - 1 / G.Tmax) / (1 / G.Tmax)
ax.hist(x, bins=25, color=BLUE, edgecolor='white', lw=0.4)
ax.set_xlabel('1/T span as fraction of distance to 1/T = 0')
ax.set_ylabel('number of pairs'); ax.grid(axis='x', visible=False)
ax.set_title('(b)', loc='left', fontsize=8.5)

fix(fig); fig.savefig('out/fig5_conditioning.png'); plt.close(fig)

# Fig 6: theta with propagated errors (Fig 6)
th2 = pd.read_csv('out/theta.csv', encoding='utf-8').sort_values('theta').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(6.6, 4.6))
for i, rr in th2.iterrows():
    ax.plot([rr.Tmin, rr.Tmax], [i, i], color=BLUE, lw=3, solid_capstyle='butt')
    inside = rr.Tmin <= rr.theta <= rr.Tmax
    ax.errorbar(rr.theta, i, xerr=rr.sig_theta, fmt='o', ms=4,
                color=AQUA if inside else ORANGE, ecolor=MUTED,
                elinewidth=0.6, capsize=1.5,
                markeredgecolor='white', markeredgewidth=0.5, zorder=3)
ax.set_yticks(range(len(th2)))
ax.set_yticklabels([f'{a} / {b}' for a, b in zip(th2.polymer, th2.solvent)], fontsize=7)
ax.set_xlabel('T, K'); ax.grid(axis='y', visible=False)
ax.legend(handles=[Line2D([], [], color=BLUE, lw=3, label='measurement range'),
                   Line2D([], [], marker='o', ls='', color=AQUA, label='θ inside range'),
                   Line2D([], [], marker='o', ls='', color=ORANGE, label='θ extrapolated'),
                   Line2D([], [], color=MUTED, lw=0.8, label='±σ(θ)')],
          fontsize=7, loc='lower right')
fix(fig); fig.savefig('out/fig6_theta_err.png'); plt.close(fig)

# Fig SI: residual compensation (collinear with 1/Thm) — SI
fig, axs = plt.subplots(1, 2, figsize=(7.4, 2.6))

G_ = G.copy()
G_['invThm'] = 1.0 / G_.Thm
a_res = G_.A - np.polyval(np.polyfit(G_.invThm, G_.A, 1), G_.invThm)
b_res = G_.B - np.polyval(np.polyfit(G_.invThm, G_.B, 1), G_.invThm)

ax = axs[0]
ax.scatter(b_res, a_res, s=18, c=BLUE, edgecolors='white', linewidths=0.6, zorder=3)
q = stats.pearsonr(b_res, a_res)
ax.set_xlabel(r'$B$ residual after removing $1/T_{hm}$')
ax.set_ylabel(r'$A$ residual after removing $1/T_{hm}$')
ax.set_title(f'(a) r = {cm(q[0])}, p = {cm(q[1], 3)}', loc='left', fontsize=8.5)

ax = axs[1]
ax.scatter(G.B, G.A, s=18, c=BLUE, edgecolors='white', linewidths=0.6, zorder=3)
xs = np.linspace(G.B.min(), G.B.max(), 50)
ax.plot(xs, G.chi_hm.mean() - xs / G.Thm.mean(), color=ORANGE, lw=1.2, zorder=2,
        label=r'$A=\chi(T_{hm})-B/T_{hm}$ (means)')
ax.set_xlabel('B, K'); ax.set_ylabel('A')
ax.set_title('(b) least-squares identity', loc='left', fontsize=8.5)
ax.legend(fontsize=7, loc='upper right')

fix(fig); fig.savefig('out/figS1_residual.png'); plt.close(fig)
print('done')

done


In [35]:
import os
for f in ['fig1_pairs.png',
          'fig3_artefact.png',
          'fig4_Tcomp.png',
          'fig5_conditioning.png',
          'fig6_theta_err.png',
          'figS1_residual.png']:
    p = os.path.join('out', f)
    print(p, os.path.exists(p), os.path.getsize(p) if os.path.exists(p) else '')

out\fig1_pairs.png True 134377
out\fig3_artefact.png True 122219
out\fig4_Tcomp.png True 105482
out\fig5_conditioning.png True 79639
out\fig6_theta_err.png True 387010
out\figS1_residual.png True 108645


In [37]:
import json, os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, ScalarFormatter, AutoMinorLocator, NullLocator
from matplotlib.lines import Line2D
from scipy import stats

try:
    sys.stdout.reconfigure(encoding='utf-8')
except Exception:
    pass

# Silence any residual Locator warnings, though the fixes below remove the cause
import warnings
warnings.filterwarnings('ignore', message='.*Locator attempting to generate.*')

W = os.path.expanduser('~/Downloads')
OUT = f'{W}/out'
FIG = f'{OUT}/fig'
os.makedirs(FIG, exist_ok=True)

DECIMAL_COMMA = False

# style
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 8,
    'axes.labelsize': 8,
    'axes.titlesize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.linewidth': 0.6,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.edgecolor': 'black',
    'xtick.color': 'black',
    'ytick.color': 'black',
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.size': 3,
    'ytick.major.size': 3,
    'xtick.minor.size': 1.5,
    'ytick.minor.size': 1.5,
    'xtick.major.width': 0.6,
    'ytick.major.width': 0.6,
    'axes.grid': False,
    'grid.color': '0.85',
    'grid.linewidth': 0.4,
    'grid.linestyle': '-',
    'axes.axisbelow': True,
    'legend.frameon': False,
    'legend.handlelength': 1.2,
    'legend.handletextpad': 0.4,
    'legend.borderpad': 0.2,
    'legend.labelspacing': 0.25,
    'savefig.dpi': 600,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.02,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})

# Okabe-Ito colour-blind-safe palette
BLUE   = '#0072B2'
ORANGE = '#D55E00'
GREEN  = '#009E73'
YELLOW = '#E69F00'
SKY    = '#56B4E9'
PURPLE = '#CC79A7'
GRAY   = '#999999'
INK    = '#000000'

def _fmt(v, p):
    s = f'{v:g}'
    if DECIMAL_COMMA:
        s = s.replace('.', ',')
    return s.replace('-', '\u2212')

FF = FuncFormatter(_fmt)
cm = lambda x, n=2: _fmt(round(float(x), n), None)

def style_ax(ax, panel=None, minor=()):
    """Apply common styling.

    Minor ticks are OFF by default. Pass minor=('x',), minor=('y',) or
    minor=('x','y') to enable them explicitly on numeric axes only.
    Never enable minor ticks on a categorical or symlog axis.
    """
    ax.xaxis.set_major_formatter(FF)
    ax.yaxis.set_major_formatter(FF)
    ax.xaxis.set_minor_locator(AutoMinorLocator() if 'x' in minor else NullLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator() if 'y' in minor else NullLocator())
    ax.tick_params(which='both', top=False, right=False)
    if panel is not None:
        ax.text(0.02, 0.98, f'({panel})', transform=ax.transAxes,
                ha='left', va='top', fontsize=9, fontweight='bold')

def save(fig, name):
    for ext in ('pdf', 'svg', 'png'):
        fig.savefig(f'{FIG}/{name}.{ext}')
    plt.close(fig)
    print('wrote', name)

# data
with open(f'{OUT}/fit.json', encoding='utf-8') as f:
    O = json.load(f)
d = pd.read_csv(f'{W}/chi_parameter.csv', encoding='utf-8')
for c in ['Polymer', 'Solvent']:
    d[c] = d[c].astype(str).str.strip()
d['Polymer'] = d.Polymer.str.replace('µ', 'ε')
d['Reference'] = d.Reference.astype(str)
F = pd.read_csv(f'{OUT}/fits.csv', encoding='utf-8')
G = F[F.R2 > 0.7].copy()
G = G[np.isfinite(G.A) & np.isfinite(G.B) & np.isfinite(G.se_A)
      & np.isfinite(G.chi_hm) & np.isfinite(G.Thm)]

SRC_MAP = {
    'Orwoll & Arnold (2007)':              ('69002', BLUE),
    'Polymer Handbook (4th ed.)':          ('Brandrup', ORANGE),
    'Gündüz & Dinçer (1980), IGC':         ('90035', GREEN),
    'other':                               (None, GRAY),
}
def src_of(r):
    for name, (key, _) in SRC_MAP.items():
        if key and key in r:
            return name
    return 'other'
SRC_COLOR = {k: v[1] for k, v in SRC_MAP.items()}

# Fig 1  three mixed-source pairs (no legend; caption explains colours)
pairs = [('polystyrene', 'cyclohexane', 'PS / cyclohexane'),
         ('polystyrene', 'toluene',     'PS / toluene'),
         ('polystyrene', 'methanol',    'PS / methanol')]

fig, axs = plt.subplots(1, 3, figsize=(7.1, 1.9), sharey=False)
for ax, (p, s, lab) in zip(axs, pairs):
    g = d[(d.Polymer == p) & (d.Solvent == s)]
    g = g.assign(src=g.Reference.map(src_of))
    for src, gg in g.groupby('src'):
        ax.scatter(gg.temperature, gg.chi, s=10, c=SRC_COLOR[src],
                   edgecolors='white', linewidths=0.3, zorder=3)
    sub = F[(F.polymer == p) & (F.solvent == s)]
    if len(sub):
        f = sub.iloc[0]
        T = np.linspace(g.temperature.min(), g.temperature.max(), 100)
        ax.plot(T, f.A + f.B / T, color=INK, lw=1.0, zorder=2)
        extra = f', B = {cm(f.B, 0)} K' if (p, s) == ('polystyrene', 'cyclohexane') else ''
        ax.set_title(f'{lab}\n$R^2$ = {cm(f.R2)}{extra}, n = {int(f.n)}',
                     loc='left', fontsize=8, pad=3)
    else:
        ax.set_title(lab, loc='left', fontsize=8, pad=3)
    ax.set_xlabel('$T$ (K)')
    style_ax(ax, minor=('x',))   # numeric T axis only
    if (p, s) == ('polystyrene', 'cyclohexane'):
        ax.axhline(0.5, color=GRAY, ls=':', lw=0.7, zorder=1)
        ax.axvline(307.6, color=GRAY, ls='--', lw=0.7, zorder=1)

axs[0].set_ylabel(r'$\chi$')
fig.subplots_adjust(bottom=0.22, wspace=0.35)
save(fig, 'fig1_pairs')

# Fig 3  A vs B, null model, chi(Thm) vs B
fig, axs = plt.subplots(1, 3, figsize=(7.1, 2.3))

# (a) A vs B
ax = axs[0]
ax.scatter(G.B, G.A, s=9, c=BLUE, edgecolors='white', linewidths=0.3, zorder=3)
r = O['repro']
xs = np.linspace(G.B.min(), G.B.max(), 100)
ax.plot(xs, r['intercept'] + r['slope'] * xs, color=INK, lw=1.0, zorder=2)
ax.set_xlabel('$B$ (K)')
ax.set_ylabel('$A$')
ax.set_title(f"$r$ = {cm(r['r'])}\n$T_{{\\rm comp}}$ = {r['Tcomp']:.0f} K",
             loc='left', fontsize=8, pad=3)
style_ax(ax, 'a', minor=('x', 'y'))

# (b) null model
ax = axs[1]
rng = np.random.default_rng(0)
rn = []
for _ in range(10000):
    ch = rng.permutation(G.chi_hm.values)
    rn.append(stats.pearsonr(G.B.values, ch - G.B.values / G.Thm.values)[0])
ax.hist(rn, bins=45, color=GRAY, edgecolor='white', lw=0.2, alpha=0.9)
ax.axvspan(O['null_r_CI'][0], O['null_r_CI'][1], color=GRAY, alpha=0.18, lw=0)
ax.axvline(r['r'], color=ORANGE, lw=1.4, zorder=4)
y_top = ax.get_ylim()[1]
ax.annotate('observed', xy=(r['r'], y_top * 0.72),
            xytext=(r['r'], y_top * 0.90),
            ha='center', va='bottom', fontsize=7, color=ORANGE,
            arrowprops=dict(arrowstyle='-', color=ORANGE, lw=0.7,
                            shrinkA=0, shrinkB=2))
ax.set_xlabel(r'$r(A, B)$ with shuffled $\chi(T_{\rm hm})$')
ax.set_ylabel('count')
ax.set_title('null model', loc='left', fontsize=8, pad=3)
style_ax(ax, 'b', minor=('x',))

# (c) chi(Thm) vs B
ax = axs[2]
ax.scatter(G.B, G.chi_hm, s=9, c=GREEN, edgecolors='white', linewidths=0.3, zorder=3)
ax.set_xlabel('$B$ (K)')
ax.set_ylabel(r'$\chi(T_{\rm hm})$')
q = stats.pearsonr(G.chi_hm, G.B)
ax.set_title(f'$r$ = {cm(q[0])}, $p$ = {cm(q[1], 3)}',
             loc='left', fontsize=8, pad=3)
style_ax(ax, 'c', minor=('x', 'y'))

fig.subplots_adjust(wspace=0.45)
save(fig, 'fig3_artefact')

# Fig 4  Tcomp vs Thm (single column)
S = pd.DataFrame(O['subsets'])
rl = O['reliable']

fig, ax = plt.subplots(figsize=(3.6, 3.0))
lo = min(S.Thm.min(), S.Tcomp.min(), rl['Thm'], rl['Tcomp']) - 8
hi = max(S.Thm.max(), S.Tcomp.max(), rl['Thm'], rl['Tcomp']) + 8
ax.plot([lo, hi], [lo, hi], color=GRAY, ls='--', lw=0.7, zorder=1)
ax.scatter(S.Thm, S.Tcomp, s=26, c=BLUE, edgecolors='white',
           linewidths=0.4, zorder=3)
ax.scatter([rl['Thm']], [rl['Tcomp']], s=30, c=ORANGE, edgecolors='white',
           linewidths=0.4, zorder=4)

lab = {'R² > 0.7 (as in draft)':            '$R^2>0.7$',
       'R² > 0.7, ΔT > 50 K':               '$R^2>0.7$, $\\Delta T>50$ K',
       'R² > 0.7, single source':           '$R^2>0.7$, single source',
       'all 143 pairs':                     'all 143 pairs',
       'all pairs, single source':          'all, single source',
       'single source, ΔT > 50 K':          'single source, $\\Delta T>50$ K'}
offsets = {'R² > 0.7 (as in draft)':       ( 7, -7),
           'R² > 0.7, single source':      ( 7,  5),
           'all 143 pairs':                (-7,  7),
           'R² > 0.7, ΔT > 50 K':          ( 7,  3),
           'all pairs, single source':     ( 7,  3),
           'single source, ΔT > 50 K':     ( 7,  3)}
for _, rr in S.iterrows():
    ox, oy = offsets.get(rr.label, (7, 3))
    ha = 'right' if rr.label == 'all 143 pairs' else 'left'
    ax.annotate(lab.get(rr.label, rr.label), (rr.Thm, rr.Tcomp),
                xytext=(ox, oy), textcoords='offset points',
                fontsize=6.5, color=INK, ha=ha, va='center')
ax.annotate(f"reliable ($n$ = {rl['n']})", (rl['Thm'], rl['Tcomp']),
            xytext=(7, -7), textcoords='offset points', fontsize=6.5,
            color=ORANGE, va='center', ha='left')

ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.set_xlabel(r'$T_{\rm hm}$ (K)')
ax.set_ylabel(r'$T_{\rm comp}$ (K)', labelpad=4)
style_ax(ax, minor=('x', 'y'))   # numeric axes, minor ticks are safe
save(fig, 'fig4_Tcomp')

# Fig 5  conditioning: sigma(A) vs |A|, 1/T span
fig, axs = plt.subplots(1, 2, figsize=(7.1, 2.4))

ax = axs[0]
ax.scatter(G.A.abs(), G.se_A, s=9, c=BLUE, edgecolors='white',
           linewidths=0.3, zorder=3)
m = max(G.A.abs().max(), G.se_A.max()) * 1.4
ax.plot([0, m], [0, m / 2], color=GRAY, ls='--', lw=0.7, zorder=1)
ax.text(m * 0.52, m * 0.33, r'$|A|=2\sigma(A)$', fontsize=6.5, color=INK)
ax.set_xscale('symlog', linthresh=0.1, linscale=0.7)
ax.set_yscale('symlog', linthresh=0.1, linscale=0.7)
ax.set_xlim(0, m); ax.set_ylim(0, m)
ax.set_xlabel(r'$|A|$')
ax.set_ylabel(r'$\sigma(A)$')
style_ax(ax, 'a', minor=())   # symlog: no manual minor ticks

ax = axs[1]
x = (1 / G.Tmin - 1 / G.Tmax) / (1 / G.Tmax)
ax.hist(x, bins=28, color=BLUE, edgecolor='white', lw=0.2, alpha=0.95)
ax.set_xlabel(r'$1/T$ span as fraction of distance to $1/T=0$')
ax.set_ylabel('number of pairs')
ax.grid(axis='x', visible=False)
style_ax(ax, 'b', minor=('x',))

fig.subplots_adjust(wspace=0.4)
save(fig, 'fig5_conditioning')

# Fig 6  theta with propagated errors
th2 = pd.read_csv(f'{OUT}/theta.csv', encoding='utf-8').sort_values('theta').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(3.8, 4.2))
for i, rr in th2.iterrows():
    ax.plot([rr.Tmin, rr.Tmax], [i, i], color=BLUE, lw=2.4,
            solid_capstyle='butt', zorder=2)
    inside = rr.Tmin <= rr.theta <= rr.Tmax
    ax.errorbar(rr.theta, i, xerr=rr.sig_theta, fmt='o', ms=3.4,
                color=GREEN if inside else ORANGE, ecolor=GRAY,
                elinewidth=0.6, capsize=1.4,
                markeredgecolor='white', markeredgewidth=0.3, zorder=3)
ax.set_yticks(range(len(th2)))
ax.set_yticklabels([str(i) for i in range(len(th2))], fontsize=6.5)
ax.yaxis.set_minor_locator(NullLocator())
ax.set_xlabel('$T$ (K)')
ax.set_ylabel('row index')
ax.tick_params(axis='y', which='minor', length=0)
ax.grid(axis='y', visible=False)
ax.legend(handles=[Line2D([], [], color=BLUE, lw=2.4, label='measured range'),
                   Line2D([], [], marker='o', ls='', color=GREEN,
                          markeredgecolor='white', markersize=4,
                          label=r'$\theta$ inside'),
                   Line2D([], [], marker='o', ls='', color=ORANGE,
                          markeredgecolor='white', markersize=4,
                          label=r'$\theta$ extrapolated'),
                   Line2D([], [], color=GRAY, lw=0.8, label=r'$\pm\sigma(\theta)$')],
          fontsize=6.2, loc='upper center', bbox_to_anchor=(0.5, -0.10),
          ncol=4, columnspacing=1.0, handletextpad=0.4)
fig.subplots_adjust(bottom=0.22)
style_ax(ax, minor=('x',))
save(fig, 'fig6_theta_err')

# Fig S1  residual compensation (SI, collinear with 1/Thm)
G_ = G.copy()
G_['invThm'] = 1.0 / G_.Thm
a_res = G_.A - np.polyval(np.polyfit(G_.invThm, G_.A, 1), G_.invThm)
b_res = G_.B - np.polyval(np.polyfit(G_.invThm, G_.B, 1), G_.invThm)

fig, axs = plt.subplots(1, 2, figsize=(7.1, 2.5))
ax = axs[0]
ax.scatter(b_res, a_res, s=9, c=BLUE, edgecolors='white',
           linewidths=0.3, zorder=3)
q = stats.pearsonr(b_res, a_res)
ax.set_xlabel(r'$B$ residual after removing $1/T_{\rm hm}$')
ax.set_ylabel(r'$A$ residual after removing $1/T_{\rm hm}$')
ax.set_title(f'$r$ = {cm(q[0])}, $p$ = {cm(q[1], 3)}',
             loc='left', fontsize=8, pad=3)
style_ax(ax, 'a', minor=('x', 'y'))

ax = axs[1]
ax.scatter(G.B, G.A, s=9, c=BLUE, edgecolors='white',
           linewidths=0.3, zorder=3)
xs = np.linspace(G.B.min(), G.B.max(), 100)
ax.plot(xs, G.chi_hm.mean() - xs / G.Thm.mean(), color=ORANGE, lw=1.0, zorder=2,
        label=r'$A=\chi(T_{\rm hm})-B/T_{\rm hm}$ (means)')
ax.set_xlabel('$B$ (K)')
ax.set_ylabel('$A$')
ax.set_title('least-squares identity', loc='left', fontsize=8, pad=3)
ax.legend(loc='upper right', fontsize=6.5)
style_ax(ax, 'b', minor=('x', 'y'))
fig.subplots_adjust(wspace=0.45)
save(fig, 'figS1_residual')

print('all figures written to', FIG)

wrote fig1_pairs
wrote fig3_artefact
wrote fig4_Tcomp
wrote fig5_conditioning
wrote fig6_theta_err
wrote figS1_residual
all figures written to C:\Users\130/Downloads/out/fig


In [38]:
import json, os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, ScalarFormatter, AutoMinorLocator, NullLocator
from matplotlib.lines import Line2D
from scipy import stats

try:
    sys.stdout.reconfigure(encoding='utf-8')
except Exception:
    pass

import warnings
warnings.filterwarnings('ignore', message='.*Locator attempting to generate.*')

W = os.path.expanduser('~/Downloads')
OUT = f'{W}/out'
FIG = f'{OUT}/fig'
os.makedirs(FIG, exist_ok=True)

DECIMAL_COMMA = False

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 8,
    'axes.labelsize': 8,
    'axes.titlesize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.linewidth': 0.6,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.edgecolor': 'black',
    'xtick.color': 'black',
    'ytick.color': 'black',
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.size': 3,
    'ytick.major.size': 3,
    'xtick.minor.size': 1.5,
    'ytick.minor.size': 1.5,
    'xtick.major.width': 0.6,
    'ytick.major.width': 0.6,
    'axes.grid': False,
    'grid.color': '0.85',
    'grid.linewidth': 0.4,
    'grid.linestyle': '-',
    'axes.axisbelow': True,
    'legend.frameon': False,
    'legend.handlelength': 1.2,
    'legend.handletextpad': 0.4,
    'legend.borderpad': 0.2,
    'legend.labelspacing': 0.25,
    'savefig.dpi': 600,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.02,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})

BLUE   = '#0072B2'
ORANGE = '#D55E00'
GREEN  = '#009E73'
YELLOW = '#E69F00'
SKY    = '#56B4E9'
PURPLE = '#CC79A7'
GRAY   = '#999999'
INK    = '#000000'

def _fmt(v, p):
    s = f'{v:g}'
    if DECIMAL_COMMA:
        s = s.replace('.', ',')
    return s.replace('-', '\u2212')

FF = FuncFormatter(_fmt)
cm = lambda x, n=2: _fmt(round(float(x), n), None)

def style_ax(ax, panel=None, minor=()):
    ax.xaxis.set_major_formatter(FF)
    ax.yaxis.set_major_formatter(FF)
    ax.xaxis.set_minor_locator(AutoMinorLocator() if 'x' in minor else NullLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator() if 'y' in minor else NullLocator())
    ax.tick_params(which='both', top=False, right=False)
    if panel is not None:
        ax.text(0.02, 0.98, f'({panel})', transform=ax.transAxes,
                ha='left', va='top', fontsize=9, fontweight='bold')

def save(fig, name):
    for ext in ('pdf', 'svg', 'png'):
        fig.savefig(f'{FIG}/{name}.{ext}')
    plt.close(fig)
    print('wrote', name)

# data
with open(f'{OUT}/fit.json', encoding='utf-8') as f:
    O = json.load(f)
d = pd.read_csv(f'{W}/chi_parameter.csv', encoding='utf-8')
for c in ['Polymer', 'Solvent']:
    d[c] = d[c].astype(str).str.strip()
d['Polymer'] = d.Polymer.str.replace('µ', 'ε')
d['Reference'] = d.Reference.astype(str)
F = pd.read_csv(f'{OUT}/fits.csv', encoding='utf-8')
G = F[F.R2 > 0.7].copy()
G = G[np.isfinite(G.A) & np.isfinite(G.B) & np.isfinite(G.se_A)
      & np.isfinite(G.chi_hm) & np.isfinite(G.Thm)]

SRC_MAP = {
    'Orwoll & Arnold (2007)':        ('69002', BLUE),
    'Polymer Handbook (4th ed.)':    ('Brandrup', ORANGE),
    'Gündüz & Dinçer (1980), IGC':   ('90035', GREEN),
    'other':                         (None, GRAY),
}
def src_of(r):
    for name, (key, _) in SRC_MAP.items():
        if key and key in r:
            return name
    return 'other'
SRC_COLOR = {k: v[1] for k, v in SRC_MAP.items()}

# Fig 1  three mixed-source pairs
pairs = [('polystyrene', 'cyclohexane', 'PS / cyclohexane'),
         ('polystyrene', 'toluene',     'PS / toluene'),
         ('polystyrene', 'methanol',    'PS / methanol')]

fig, axs = plt.subplots(1, 3, figsize=(7.1, 1.9), sharey=False)
for ax, (p, s, lab) in zip(axs, pairs):
    g = d[(d.Polymer == p) & (d.Solvent == s)]
    g = g.assign(src=g.Reference.map(src_of))
    for src, gg in g.groupby('src'):
        ax.scatter(gg.temperature, gg.chi, s=10, c=SRC_COLOR[src],
                   edgecolors='white', linewidths=0.3, zorder=3)
    sub = F[(F.polymer == p) & (F.solvent == s)]
    if len(sub):
        f = sub.iloc[0]
        T = np.linspace(g.temperature.min(), g.temperature.max(), 100)
        ax.plot(T, f.A + f.B / T, color=INK, lw=1.0, zorder=2)
        extra = f', B = {cm(f.B, 0)} K' if (p, s) == ('polystyrene', 'cyclohexane') else ''
        ax.set_title(f'{lab}\n$R^2$ = {cm(f.R2)}{extra}, n = {int(f.n)}',
                     loc='left', fontsize=8, pad=3)
    else:
        ax.set_title(lab, loc='left', fontsize=8, pad=3)
    ax.set_xlabel('$T$ (K)')
    style_ax(ax, minor=('x',))
    if (p, s) == ('polystyrene', 'cyclohexane'):
        ax.axhline(0.5, color=GRAY, ls=':', lw=0.7, zorder=1)
        ax.axvline(307.6, color=GRAY, ls='--', lw=0.7, zorder=1)

axs[0].set_ylabel(r'$\chi$')
fig.subplots_adjust(bottom=0.22, wspace=0.35)
save(fig, 'fig1_pairs')

# Fig 3  A vs B, null model, chi(Thm) vs B
fig, axs = plt.subplots(1, 3, figsize=(7.1, 2.3))

ax = axs[0]
ax.scatter(G.B, G.A, s=9, c=BLUE, edgecolors='white', linewidths=0.3, zorder=3)
r = O['repro']
xs = np.linspace(G.B.min(), G.B.max(), 100)
ax.plot(xs, r['intercept'] + r['slope'] * xs, color=INK, lw=1.0, zorder=2)
ax.set_xlabel('$B$ (K)')
ax.set_ylabel('$A$')
ax.set_title(f"$r$ = {cm(r['r'])}\n$T_{{\\rm comp}}$ = {r['Tcomp']:.0f} K",
             loc='left', fontsize=8, pad=3)
style_ax(ax, 'a', minor=('x', 'y'))

ax = axs[1]
rng = np.random.default_rng(0)
rn = []
for _ in range(10000):
    ch = rng.permutation(G.chi_hm.values)
    rn.append(stats.pearsonr(G.B.values, ch - G.B.values / G.Thm.values)[0])
ax.hist(rn, bins=45, color=GRAY, edgecolor='white', lw=0.2, alpha=0.9)
ax.axvspan(O['null_r_CI'][0], O['null_r_CI'][1], color=GRAY, alpha=0.18, lw=0)
ax.axvline(r['r'], color=ORANGE, lw=1.4, zorder=4)
y_top = ax.get_ylim()[1]
ax.annotate('observed', xy=(r['r'], y_top * 0.72),
            xytext=(r['r'], y_top * 0.90),
            ha='center', va='bottom', fontsize=7, color=ORANGE,
            arrowprops=dict(arrowstyle='-', color=ORANGE, lw=0.7,
                            shrinkA=0, shrinkB=2))
ax.set_xlabel(r'$r(A, B)$ with shuffled $\chi(T_{\rm hm})$')
ax.set_ylabel('count')
ax.set_title('null model', loc='left', fontsize=8, pad=3)
style_ax(ax, 'b', minor=('x',))

ax = axs[2]
ax.scatter(G.B, G.chi_hm, s=9, c=GREEN, edgecolors='white', linewidths=0.3, zorder=3)
ax.set_xlabel('$B$ (K)')
ax.set_ylabel(r'$\chi(T_{\rm hm})$')
q = stats.pearsonr(G.chi_hm, G.B)
ax.set_title(f'$r$ = {cm(q[0])}, $p$ = {cm(q[1], 3)}',
             loc='left', fontsize=8, pad=3)
style_ax(ax, 'c', minor=('x', 'y'))

fig.subplots_adjust(wspace=0.45)
save(fig, 'fig3_artefact')

# Fig 4  Tcomp vs Thm
S = pd.DataFrame(O['subsets'])
rl = O['reliable']

fig, ax = plt.subplots(figsize=(3.6, 3.0))
lo = min(S.Thm.min(), S.Tcomp.min(), rl['Thm'], rl['Tcomp']) - 8
hi = max(S.Thm.max(), S.Tcomp.max(), rl['Thm'], rl['Tcomp']) + 8
ax.plot([lo, hi], [lo, hi], color=GRAY, ls='--', lw=0.7, zorder=1)
ax.scatter(S.Thm, S.Tcomp, s=26, c=BLUE, edgecolors='white',
           linewidths=0.4, zorder=3)
ax.scatter([rl['Thm']], [rl['Tcomp']], s=30, c=ORANGE, edgecolors='white',
           linewidths=0.4, zorder=4)

lab = {'R² > 0.7 (as in draft)':  '$R^2>0.7$',
       'R² > 0.7, ΔT > 50 K':     '$R^2>0.7$, $\\Delta T>50$ K',
       'R² > 0.7, single source': '$R^2>0.7$, single source',
       'all 143 pairs':           'all 143 pairs',
       'all pairs, single source':'all, single source',
       'single source, ΔT > 50 K':'single source, $\\Delta T>50$ K'}
offsets = {'R² > 0.7 (as in draft)':  ( 7, -7),
           'R² > 0.7, single source': ( 7,  5),
           'all 143 pairs':           (-7,  7),
           'R² > 0.7, ΔT > 50 K':     ( 7,  3),
           'all pairs, single source':( 7,  3),
           'single source, ΔT > 50 K':( 7,  3)}
for _, rr in S.iterrows():
    ox, oy = offsets.get(rr.label, (7, 3))
    ha = 'right' if rr.label == 'all 143 pairs' else 'left'
    ax.annotate(lab.get(rr.label, rr.label), (rr.Thm, rr.Tcomp),
                xytext=(ox, oy), textcoords='offset points',
                fontsize=6.5, color=INK, ha=ha, va='center')
ax.annotate(f"reliable ($n$ = {rl['n']})", (rl['Thm'], rl['Tcomp']),
            xytext=(7, -7), textcoords='offset points', fontsize=6.5,
            color=ORANGE, va='center', ha='left')

ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.set_xlabel(r'$T_{\rm hm}$ (K)')
ax.set_ylabel(r'$T_{\rm comp}$ (K)', labelpad=4)
style_ax(ax, minor=('x', 'y'))
save(fig, 'fig4_Tcomp')

# Fig 5  conditioning: sigma(A) vs |A|, 1/T span
fig, axs = plt.subplots(1, 2, figsize=(7.1, 2.4))

ax = axs[0]
ax.scatter(G.A.abs(), G.se_A, s=9, c=BLUE, edgecolors='white',
           linewidths=0.3, zorder=3)
m = max(G.A.abs().max(), G.se_A.max()) * 1.4
ax.plot([0, m], [0, m / 2], color=GRAY, ls='--', lw=0.7, zorder=1)
ax.text(m * 0.52, m * 0.33, r'$|A|=2\sigma(A)$', fontsize=6.5, color=INK)
ax.set_xscale('symlog', linthresh=0.1, linscale=0.7)
ax.set_yscale('symlog', linthresh=0.1, linscale=0.7)
ax.set_xlim(0, m); ax.set_ylim(0, m)
ax.set_xlabel(r'$|A|$')
ax.set_ylabel(r'$\sigma(A)$')
style_ax(ax, 'a', minor=())

ax = axs[1]
x = (1 / G.Tmin - 1 / G.Tmax) / (1 / G.Tmax)
ax.hist(x, bins=28, color=BLUE, edgecolor='white', lw=0.2, alpha=0.95)
ax.set_xlabel(r'$1/T$ span as fraction of distance to $1/T=0$')
ax.set_ylabel('number of pairs')
ax.grid(axis='x', visible=False)
style_ax(ax, 'b', minor=('x',))

fig.subplots_adjust(wspace=0.4)
save(fig, 'fig5_conditioning')

# Fig 6  theta with propagated errors
th2 = pd.read_csv(f'{OUT}/theta.csv', encoding='utf-8').sort_values('theta').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(3.8, 4.2))
for i, rr in th2.iterrows():
    ax.plot([rr.Tmin, rr.Tmax], [i, i], color=BLUE, lw=2.4,
            solid_capstyle='butt', zorder=2)
    inside = rr.Tmin <= rr.theta <= rr.Tmax
    ax.errorbar(rr.theta, i, xerr=rr.sig_theta, fmt='o', ms=3.4,
                color=GREEN if inside else ORANGE, ecolor=GRAY,
                elinewidth=0.6, capsize=1.4,
                markeredgecolor='white', markeredgewidth=0.3, zorder=3)
ax.set_yticks(range(len(th2)))
ax.set_yticklabels([str(i) for i in range(len(th2))], fontsize=6.5)
ax.yaxis.set_minor_locator(NullLocator())
ax.set_xlabel('$T$ (K)')
ax.set_ylabel('row index')
ax.tick_params(axis='y', which='minor', length=0)
ax.grid(axis='y', visible=False)
ax.legend(handles=[Line2D([], [], color=BLUE, lw=2.4, label='measured range'),
                   Line2D([], [], marker='o', ls='', color=GREEN,
                          markeredgecolor='white', markersize=4,
                          label=r'$\theta$ inside'),
                   Line2D([], [], marker='o', ls='', color=ORANGE,
                          markeredgecolor='white', markersize=4,
                          label=r'$\theta$ extrapolated'),
                   Line2D([], [], color=GRAY, lw=0.8, label=r'$\pm\sigma(\theta)$')],
          fontsize=6.2, loc='upper center', bbox_to_anchor=(0.5, -0.10),
          ncol=4, columnspacing=1.0, handletextpad=0.4)
fig.subplots_adjust(bottom=0.22)
style_ax(ax, minor=('x',))
save(fig, 'fig6_theta_err')

# Fig S1  residual compensation (SI, collinear with 1/Thm)
G_ = G.copy()
G_['invThm'] = 1.0 / G_.Thm
a_res = G_.A - np.polyval(np.polyfit(G_.invThm, G_.A, 1), G_.invThm)
b_res = G_.B - np.polyval(np.polyfit(G_.invThm, G_.B, 1), G_.invThm)

fig, axs = plt.subplots(1, 2, figsize=(7.1, 2.5))
ax = axs[0]
ax.scatter(b_res, a_res, s=9, c=BLUE, edgecolors='white',
           linewidths=0.3, zorder=3)
q = stats.pearsonr(b_res, a_res)
ax.set_xlabel(r'$B$ residual after removing $1/T_{\rm hm}$')
ax.set_ylabel(r'$A$ residual after removing $1/T_{\rm hm}$')
ax.set_title(f'$r$ = {cm(q[0])}, $p$ = {cm(q[1], 3)}',
             loc='left', fontsize=8, pad=3)
style_ax(ax, 'a', minor=('x', 'y'))

ax = axs[1]
ax.scatter(G.B, G.A, s=9, c=BLUE, edgecolors='white',
           linewidths=0.3, zorder=3)
xs = np.linspace(G.B.min(), G.B.max(), 100)
ax.plot(xs, G.chi_hm.mean() - xs / G.Thm.mean(), color=ORANGE, lw=1.0, zorder=2,
        label=r'$A=\chi(T_{\rm hm})-B/T_{\rm hm}$ (means)')
ax.set_xlabel('$B$ (K)')
ax.set_ylabel('$A$')
ax.set_title('least-squares identity', loc='left', fontsize=8, pad=3)
ax.legend(loc='upper right', fontsize=6.5)
style_ax(ax, 'b', minor=('x', 'y'))
fig.subplots_adjust(wspace=0.45)
save(fig, 'figS1_residual')

print('all figures written to', FIG)

wrote fig1_pairs
wrote fig3_artefact
wrote fig4_Tcomp
wrote fig5_conditioning
wrote fig6_theta_err
wrote figS1_residual
all figures written to C:\Users\130/Downloads/out/fig


In [39]:
import os
for f in ['fig1_pairs', 'fig3_artefact', 'fig4_Tcomp',
          'fig5_conditioning', 'fig6_theta_err', 'figS1_residual']:
    for ext in ('pdf', 'svg', 'png'):
        p = f'out/fig/{f}.{ext}'
        print(p, os.path.exists(p), os.path.getsize(p) if os.path.exists(p) else '')

out/fig/fig1_pairs.pdf True 40118
out/fig/fig1_pairs.svg True 50365
out/fig/fig1_pairs.png True 165015
out/fig/fig3_artefact.pdf True 52171
out/fig/fig3_artefact.svg True 81712
out/fig/fig3_artefact.png True 245366
out/fig/fig4_Tcomp.pdf True 33010
out/fig/fig4_Tcomp.svg True 29200
out/fig/fig4_Tcomp.png True 164682
out/fig/fig5_conditioning.pdf True 42899
out/fig/fig5_conditioning.svg True 40731
out/fig/fig5_conditioning.png True 155593
out/fig/fig6_theta_err.pdf True 43286
out/fig/fig6_theta_err.svg True 72979
out/fig/fig6_theta_err.png True 170922
out/fig/figS1_residual.pdf True 49403
out/fig/figS1_residual.svg True 68205
out/fig/figS1_residual.png True 230620


In [40]:
for ext in ('pdf', 'svg', 'png'):
    p = f'out/en/fig_signal2.{ext}'
    print(p, os.path.exists(p), os.path.getsize(p) if os.path.exists(p) else '')

out/en/fig_signal2.pdf True 46381
out/en/fig_signal2.svg True 92591
out/en/fig_signal2.png True 351972


In [41]:
from PIL import Image
import os

for f in ['fig1_pairs', 'fig3_artefact', 'fig4_Tcomp',
          'fig5_conditioning', 'fig6_theta_err', 'figS1_residual']:
    p = f'out/fig/{f}.png'
    if not os.path.exists(p):
        print(p, 'MISSING')
        continue
    im = Image.open(p)
    print(f, im.size, f'{os.path.getsize(p)/1024:.0f} KB')

# Fig. 2 lives in out/en/
for ext in ('pdf', 'svg', 'png'):
    p = f'out/en/fig_signal2.{ext}'
    print(p, os.path.exists(p),
          f'{os.path.getsize(p)/1024:.0f} KB' if os.path.exists(p) else '')

fig1_pairs (3529, 1118) 161 KB
fig3_artefact (3556, 1417) 240 KB
fig4_Tcomp (1936, 1593) 161 KB
fig5_conditioning (3532, 1319) 152 KB
fig6_theta_err (1973, 1953) 167 KB
figS1_residual (3521, 1436) 225 KB
out/en/fig_signal2.pdf True 45 KB
out/en/fig_signal2.svg True 90 KB
out/en/fig_signal2.png True 344 KB
